In [6]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI

In [3]:
class CategoryEmail(TypedDict):
    job_category: str
    meeting_category: str
    spam_category: str
    bank_category: str
    email: str

In [4]:
model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9)

In [5]:
def job_category(state: CategoryEmail) -> CategoryEmail:
    return {"job_category": f"{state['email']} comes inside job category"}
    
def meeting_category(state: CategoryEmail) -> CategoryEmail:
    return {"meeting_category": f"{state['email']} comes inside meeting category"}

def spam_category(state: CategoryEmail) -> CategoryEmail:
    return {"spam_category": f"{state['email']} comes inside spam category"}

def bank_category(state: CategoryEmail) -> CategoryEmail:
    return {"bank_category": f"{state['email']} comes inside bank category"}

In [8]:
def condition_check(state: CategoryEmail) -> Literal["job_category", "meeting_category", "spam_category", "bank_category"]:
    prompt = f"""
    Classify the following email into one of the following categories: job_category, meeting_category, spam_category, bank_category.
    Email: {state['email']}
    """
    response = model(prompt)
    if "job" in response.lower():
        return "job_category"
    elif "meeting" in response.lower():
        return "meeting_category"
    elif "spam" in response.lower():
        return "spam_category"
    return "bank_category"

In [11]:
graph = StateGraph(CategoryEmail)

graph.add_node("job_category", job_category)
graph.add_node("meeting_category", meeting_category)
graph.add_node("spam_category", spam_category)
graph.add_node("bank_category", bank_category)


graph.add_conditional_edges(START, condition_check)
graph.add_edge(START, "job_category")
graph.add_edge(START, "meeting_category")
graph.add_edge(START, "spam_category")
graph.add_edge(START, "bank_category")
graph.add_edge("job_category", END)
graph.add_edge("meeting_category", END)
graph.add_edge("spam_category", END)
graph.add_edge("bank_category", END)

workflow = graph.compile()
workflow

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [ ]:
workflow.invoke({"email": "You have a meeting scheduled for tomorrow at 10 AM."})

In [ ]:
workflow.invoke({"email": "You have selected for next round of an Interview."})

In [ ]:
workflow.invoke({"email": "You got 1 CR lottary. Fill this given details to claim your prize."})

In [ ]:
workflow.invoke({"email": "10k Money is debitated from your HDFC bank account. Please check your account for more details."})